# 실습 문제 모범답안 - 340. Skills Pattern

`340_Skills_Pattern` 노트북 실습 문제(사내 규정 안내 스킬 만들기)에 대한 모범답안 예시입니다.

### 구현 요약

본문의 Skills Pattern(Progressive Disclosure)을 새로운 도메인에 그대로 적용합니다.

1. 사내 규정 스킬 2개 정의: `vacation_policy`, `expense_policy`
2. 스킬 content를 반환하는 `load_company_skill` 도구
3. 스킬 설명만 시스템 프롬프트에 추가하는 미들웨어 + 에이전트 생성 후 테스트

In [6]:
from dotenv import load_dotenv

load_dotenv()

True

In [7]:
from langchain.chat_models import init_chat_model

model = init_chat_model("gpt-5-mini", model_provider="openai")

## 1. 사내 규정 스킬 2개 정의

본문의 `Skill` 구조(name, description, content)를 그대로 사용합니다.
description은 시스템 프롬프트에 노출되고, content는 도구 호출 시에만 로드됩니다.

In [8]:
from typing import TypedDict

class Skill(TypedDict):
    """에이전트가 필요할 때 로딩할 수 있는 스킬 정의 구조"""
    name: str
    description: str
    content: str

MY_SKILLS: list[Skill] = [
    {
        "name": "vacation_policy",
        "description": "휴가 규정 (연차, 반차, 신청 절차).",
        "content": """# 휴가 규정
- 연차: 1년 근속 시 15일 부여, 최대 25일까지 누적
- 반차: 오전/오후 반차 사용 가능 (0.5일 차감)
- 신청: 사용 3일 전까지 인사 시스템에서 신청, 팀장 승인 필요
""",
    },
    {
        "name": "expense_policy",
        "description": "경비 처리 규정 (식대, 교통비, 영수증).",
        "content": """# 경비 처리 규정
- 식대: 1인 1식 15,000원 한도 (야근 시 20,000원)
- 교통비: 대중교통 실비 정산, 택시는 사전 승인 필요
- 영수증: 사용일로부터 7일 이내 제출 필수
""",
    },
]

## 2. load_company_skill 도구

본문의 `load_skill`과 같은 구조입니다.
스킬 이름으로 검색해서 content를 반환하고, 없으면 사용 가능한 스킬 목록을 안내합니다.

In [9]:
from langchain.tools import tool

@tool
def load_company_skill(skill_name: str) -> str:
    """사내 규정 스킬의 전체 내용을 로드합니다."""
    for skill in MY_SKILLS:
        if skill["name"] == skill_name:
            return f"Loaded skill: {skill_name}\n\n{skill['content']}"

    available = ", ".join(s["name"] for s in MY_SKILLS)
    return f"스킬 '{skill_name}'을 찾을 수 없습니다. 사용 가능: {available}"


## 3. 미들웨어 + 에이전트 생성 및 테스트

본문의 `SkillMiddleware`와 같은 구조로, 시스템 프롬프트에 **스킬 설명만** 추가합니다.
질문하면 에이전트가 `load_company_skill(vacation_policy)`을 호출한 뒤 규정 내용으로 답변합니다.

In [10]:
from typing import Callable
from langchain.agents import create_agent
from langchain.agents.middleware import AgentMiddleware, ModelRequest, ModelResponse

class CompanySkillMiddleware(AgentMiddleware):
    """시스템 프롬프트에 사내 규정 스킬 설명 목록을 주입하는 미들웨어"""
    tools = [load_company_skill]

    def __init__(self):
        skills_list = [f"- **{s['name']}**: {s['description']}" for s in MY_SKILLS]
        self.skills_prompt = "\n".join(skills_list)

    def wrap_model_call(
        self,
        request: ModelRequest,
        handler: Callable[[ModelRequest], ModelResponse],
    ) -> ModelResponse:
        skills_addendum = (
            f"\n\n## 사용 가능한 스킬\n\n{self.skills_prompt}\n\n"
            "질문에 답하기 전에 load_company_skill 도구로 관련 규정의 상세 내용을 로드하세요."
        )
        current = getattr(request, "system_prompt", None) or ""
        return handler(request.override(system_prompt=current + skills_addendum))


agent = create_agent(
    model,
    system_prompt="당신은 사내 규정 안내 도우미입니다. 규정에 근거해서 답변하세요.",
    middleware=[CompanySkillMiddleware()],
)

In [11]:
from langchain_core.messages import HumanMessage

# stream_mode="values" 로 새로 생성되는 메시지만 출력
for step in agent.stream(
    {"messages": [HumanMessage(content="연차 휴가는 며칠인가요?")]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

연차 휴가는 며칠인가요?
================================== Ai Message ==================================
Tool Calls:
  load_company_skill (call_mcB12heZDeTaHFPoo96NaoSy)
 Call ID: call_mcB12heZDeTaHFPoo96NaoSy
  Args:
    skill_name: vacation_policy
================================= Tool Message =================================
Name: load_company_skill

Loaded skill: vacation_policy

# 휴가 규정
- 연차: 1년 근속 시 15일 부여, 최대 25일까지 누적
- 반차: 오전/오후 반차 사용 가능 (0.5일 차감)
- 신청: 사용 3일 전까지 인사 시스템에서 신청, 팀장 승인 필요

================================== Ai Message ==================================

연차는 1년 근속 시 15일 부여됩니다. 최대 25일까지 누적 가능하고, 반차(오전/오후)는 0.5일로 사용할 수 있습니다. 사용 시 인사 시스템에서 사용 3일 전까지 신청하고 팀장 승인 필요합니다.


## 확인 포인트

- 시스템 프롬프트에는 두 스킬의 **설명(한 줄)** 만 들어가 있습니다.
- 에이전트가 질문을 보고 `load_company_skill("vacation_policy")`을 호출하면,
  그때서야 휴가 규정 상세 내용이 컨텍스트에 로드됩니다 — 이것이 **Progressive Disclosure**입니다.
- 경비 관련 질문(예: "야근 식대 한도는?")을 던지면 `expense_policy` 스킬이 로드되는 것도 확인해 보세요.